# gubapost(股吧帖子) baseline 推理 + stock-day 聚合

把 `gubapost` 语料(2020–2023)全部过原始二分类模型(`chinese-wwm-roberta.ckpt`), 得到逐帖
`class_1_prob`(看多情绪概率), 按 **stock-day(available_date × symbol)取 mean** 聚合成 baseline 因子。

**数据**: `/home/intern_fjq_2026/data/NLP/gubapost/<year>/gubapostYYYYMMDD.parquet`,
列 `baName/postId/title/content/publishTime/date/available_date`; `available_date` 已对齐下一交易日, 直接沿用。

**运行顺序**(本 notebook 自上而下):

| Cell | 作用 | 耗时 |
|---|---|---|
| 1. 配置 | 路径 / 年份 / **GPUS 开关(1 或 0,1)** / worker 数 | 秒级 |
| 2. 文件清单 | 扫描 + 行数统计, 估 ETA | ~1 min |
| 3. 启动推理 | 幂等: 已在跑/已完成会跳过 | 秒级(后台跑数小时) |
| 4. (可选)停止 | 中断后台推理 | 秒级 |
| 5. 进度监控 | 反复执行查看进度/ETA | 秒级 |
| 5b. 实时进度条 | ipywidgets 每 5s 自动刷新(可选) | 后台持续 |
| 6. stock-day 聚合 | 推理**全部完成后**执行, 流式 mean | ~5 min |
| 7. 结果体检 | 分布/覆盖/样例/图 | 秒级 |

**产物**(均在 `artifacts/gubapost_baseline/`):

- `probs_daily/gubapost_probs_YYYYMMDD.parquet` — 逐帖推理结果
  (`post_id, symbol, date, available_date, class_0_prob, class_1_prob`)
- `gubapost_stockday_mean_2020_2023.parquet` — stock-day 聚合结果
  (`available_date, symbol, sentiment_mean, n_posts`)
- `manifest.json` / `completed.jsonl` / `failed.jsonl` / `logs/` — 运行元信息与断点续跑依据

**性能参考**(A100 80G, fp16, max_length=128, 长度分桶 batch): 单卡(cuda:1, 4 worker)
实测聚合约 1.3–1.5 万条/s; `GPUS="0,1"` + `WORKERS=8` 双卡约快一倍。2020–2023 约 2 亿帖 →
单卡约 4–5 h, 双卡约 2–2.5 h。断点续跑: 已完成的日文件自动跳过, 重跑 Cell 3 即可。

In [ ]:
# 1. 配置 ----------------------------------------------------------------
import os, sys

ROOT = "/home/intern_fjq_2026/Projects/chinese-wwm-roberta"
INPUT_DIR = "/home/intern_fjq_2026/data/NLP/gubapost"
OUT_DIR = os.path.join(ROOT, "artifacts", "gubapost_baseline")
SCRIPT = os.path.join(ROOT, "scripts", "infer_gubapost.py")

YEARS = ["2020", "2021", "2022", "2023"]   # 参与推理的年份
GPUS = "1"          # ★ 显卡开关: "1" = 只用 cuda:1; "0,1" = 双卡(约快一倍)
WORKERS = 4         # worker 进程总数(在 GPUS 间均分); 双卡建议 8
TOKEN_WORKERS = 4   # 每个 worker 内部 tokenize 进程数
BATCH_TOKENS = 131072
BATCH_ROWS = 16384
MAX_LENGTH = 128    # 与原模型用法一致
DTYPE = "fp16"

os.makedirs(OUT_DIR, exist_ok=True)

# 环境体检
import torch, pandas as pd, pyarrow as pa
print("python:", sys.version.split()[0], "| torch:", torch.__version__,
      "| pandas:", pd.__version__, "| pyarrow:", pa.__version__)
print("CUDA 可用:", torch.cuda.is_available(), "| GPU 数:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU{i}: {torch.cuda.get_device_name(i)}, "
          f"空闲 {free/2**30:.0f}G / {total/2**30:.0f}G")
print()
print(f"配置: years={YEARS}, gpus={GPUS}, workers={WORKERS}, "
      f"max_length={MAX_LENGTH}, dtype={DTYPE}")
print("输出目录:", OUT_DIR)

In [ ]:
# 2. 文件清单 + 行数统计 ---------------------------------------------------
# 用 parquet 元数据并行读行数(不加载正文), 估总帖量与 ETA
import glob, re
from concurrent.futures import ProcessPoolExecutor
import pyarrow.parquet as pq

def n_rows(path):
    try:
        return path, pq.ParquetFile(path).metadata.num_rows
    except Exception:
        return path, -1

files = []
for y in YEARS:
    files += sorted(glob.glob(os.path.join(INPUT_DIR, y, "gubapost*.parquet")))
files = [f for f in files if re.fullmatch(r"gubapost\d{8}\.parquet", os.path.basename(f))]
assert files, "没有找到任何日文件, 检查 YEARS / INPUT_DIR"

with ProcessPoolExecutor(min(16, os.cpu_count())) as ex:
    counted = list(ex.map(n_rows, files))
inv = pd.DataFrame([{"path": p, "n_rows": n,
                     "date": re.search(r"(\d{8})\.parquet$", p).group(1),
                     "year": re.search(r"/(\d{4})/", p).group(1)} for p, n in counted])
inv = inv.sort_values("date").reset_index(drop=True)
bad = inv[inv.n_rows < 0]
assert bad.empty, f"以下文件元数据读取失败:\n{bad}"
inv.to_parquet(os.path.join(OUT_DIR, "inventory.parquet"), index=False)

TOTAL_POSTS = int(inv.n_rows.sum())
print(f"日文件: {len(inv)} 个 ({inv.date.iloc[0]} ~ {inv.date.iloc[-1]})")
print("逐年帖子数:")
print(inv.groupby("year").n_rows.agg(["sum", "count"]).rename(
    columns={"sum": "帖子数", "count": "文件数"}).to_string())
print(f"总帖量: {TOTAL_POSTS:,}")
est = TOTAL_POSTS / 14_000 / 3600
print(f"预计耗时: 单卡(GPUS='1')约 {est:.1f} h; 双卡(GPUS='0,1')约 {est/2:.1f} h "
      "(按 1.4 万条/s/卡)")

In [ ]:
# 3. 启动推理(幂等) --------------------------------------------------------
# 后台 detached 运行 scripts/infer_gubapost.py; 已在跑或已完成则跳过
import json, subprocess

MANIFEST = os.path.join(OUT_DIR, "manifest.json")
PID_FILE = os.path.join(OUT_DIR, "run.pid")

def _running_pid():
    if not os.path.exists(PID_FILE):
        return None
    try:
        pid = int(open(PID_FILE).read().strip())
        with open(f"/proc/{pid}/cmdline") as f:
            cmd = f.read()
        return pid if "infer_gubapost" in cmd else None
    except (FileNotFoundError, ProcessLookupError, ValueError):
        return None

done_already = False
if os.path.exists(MANIFEST):
    m = json.load(open(MANIFEST))
    if m.get("status") == "completed":
        print(f"推理已完成(manifest status=completed, {m['n_rows_inferred']:,} 行), 跳过启动。")
        done_already = True

if not done_already:
    pid = _running_pid()
    if pid:
        print(f"推理已在后台运行(pid={pid}), 跳过。监控请执行 Cell 5; 停止请执行 Cell 4。")
    else:
        cmd = [sys.executable, SCRIPT,
               "--input-dir", INPUT_DIR, "--out-dir", OUT_DIR,
               "--gpus", GPUS, "--workers", str(WORKERS),
               "--token-workers", str(TOKEN_WORKERS),
               "--batch-tokens", str(BATCH_TOKENS), "--batch-rows", str(BATCH_ROWS),
               "--max-length", str(MAX_LENGTH), "--dtype", DTYPE,
               "--years", ",".join(YEARS)]
        logf = open(os.path.join(OUT_DIR, "run.log"), "a")
        env = os.environ.copy()
        p = subprocess.Popen(cmd, stdout=logf, stderr=subprocess.STDOUT,
                             cwd=ROOT, env=env, start_new_session=True)
        with open(PID_FILE, "w") as f:
            f.write(str(p.pid))
        print(f"已启动后台推理 pid={p.pid} (gpus={GPUS}, workers={WORKERS})")
        print("  驱动日志:", os.path.join(OUT_DIR, "run.log"))
        print("  worker 日志:", os.path.join(OUT_DIR, "logs", "worker*.log"))
        print("  反复执行 Cell 5 查看进度; 预计数小时, 期间可关闭 jupyter 页面。")

In [ ]:
# 4. (可选)停止推理 ---------------------------------------------------------
# 给驱动进程发 SIGTERM, 驱动会连带终止所有 GPU worker; 已完成日文件保留, 可续跑
import os, signal
pid_file = os.path.join(OUT_DIR, "run.pid")
if os.path.exists(pid_file):
    pid = int(open(pid_file).read().strip())
    os.kill(pid, signal.SIGTERM)
    print(f"已向驱动进程 {pid} 发送 SIGTERM, worker 将被终止。")
    os.remove(pid_file)
else:
    print("没有 run.pid, 无正在运行的后台推理。")

In [ ]:
# 5. 进度监控(反复执行) ------------------------------------------------------
import glob, json, os, time

pid_file = os.path.join(OUT_DIR, "run.pid")
if os.path.exists(pid_file):
    try:
        pid = int(open(pid_file).read().strip())
        print(f"后台推理运行中: pid={pid}")
    except ValueError:
        pass

exp_files = len(inv)
done_log = os.path.join(OUT_DIR, "completed.jsonl")
inv_names = {os.path.basename(p) for p in inv.path}
done, rows = {}, 0
if os.path.exists(done_log):
    for line in open(done_log):
        d = json.loads(line)
        if d["file"] in inv_names and d["file"] not in done:
            done[d["file"]] = d
            rows += d["rows"]
have = len(glob.glob(os.path.join(OUT_DIR, "probs_daily", "gubapost_probs_*.parquet")))
print(f"进度: 完成 {len(done)}/{exp_files} 个日文件, {rows:,}/{TOTAL_POSTS:,} 帖 "
      f"({rows/max(TOTAL_POSTS,1)*100:.1f}%)")

fail_log = os.path.join(OUT_DIR, "failed.jsonl")
if os.path.exists(fail_log):
    fails = [json.loads(l) for l in open(fail_log)]
    if fails:
        print(f"[注意] 失败 {len(fails)} 个文件(重跑 Cell 3 可自动重试):",
              [f["file"] for f in fails][:10])

t0 = os.path.getmtime(done_log) if os.path.exists(done_log) else None
t_start = os.path.getmtime(pid_file) if os.path.exists(pid_file) else t0
if done and t_start:
    rate = rows / max(time.time() - t_start, 1)
    eta = (TOTAL_POSTS - rows) / max(rate, 1) / 3600
    print(f"均速 {rate:,.0f} 帖/s, 预计剩余 {eta:.1f} h")

for lf in sorted(glob.glob(os.path.join(OUT_DIR, "logs", "worker*.log"))):
    lines = open(lf).read().strip().splitlines()
    print(f"--- {os.path.basename(lf)} (末2行) ---")
    for l in lines[-2:]:
        print("   ", l)

os.system("nvidia-smi --query-gpu=index,memory.used,utilization.gpu --format=csv,noheader")

In [ ]:
# 5b. 实时进度条(可选; 重新运行本 cell 可重启刷新) ------------------------------
# 后台线程每 5s 刷新进度条, 不阻塞内核; 推理结束后停在 100%。
# 停止刷新: 执行 _guba_watch_stop.set()
import glob, json, os, threading, time
import ipywidgets as W
from IPython.display import display

if "_guba_watch_stop" in globals():
    _guba_watch_stop.set()          # 重跑本 cell 时先停掉旧线程
_guba_watch_stop = threading.Event()

inv_names = {os.path.basename(p) for p in inv.path}
pid_file = os.path.join(OUT_DIR, "run.pid")
t_start = os.path.getmtime(pid_file) if os.path.exists(pid_file) else None

bar = W.IntProgress(min=0, max=TOTAL_POSTS, description="帖子:")
label = W.HTML()
display(W.VBox([bar, label]))

def _watch():
    while not _guba_watch_stop.is_set():
        rows = files = 0
        done_log = os.path.join(OUT_DIR, "completed.jsonl")
        if os.path.exists(done_log):
            seen = {}
            for line in open(done_log):
                d = json.loads(line)
                if d["file"] in inv_names and d["file"] not in seen:
                    seen[d["file"]] = d
            files, rows = len(seen), sum(v["rows"] for v in seen.values())
        bar.value = min(rows, TOTAL_POSTS)
        msg = (f"完成 {files}/{len(inv)} 个日文件, "
               f"{rows:,}/{TOTAL_POSTS:,} 帖 ({rows/max(TOTAL_POSTS,1)*100:.1f}%)")
        if t_start and rows:
            rate = rows / max(time.time() - t_start, 1)
            msg += f" | 均速 {rate:,.0f} 帖/s | 剩余约 {(TOTAL_POSTS-rows)/max(rate,1)/3600:.1f} h"
        label.value = msg
        _guba_watch_stop.wait(5)

threading.Thread(target=_watch, daemon=True).start()
print("进度条已启动(每 5s 刷新)。")

In [ ]:
# 6. stock-day 聚合(mean) —— 推理全部完成后执行 -------------------------------
# 流式: 逐日文件 groupby 局部和, 最后一次合并求 mean, 内存占用小
import glob, os, re, time

expected = {d for d in inv.date}
have = {re.search(r"gubapost_probs_(\d{8})\.parquet$", os.path.basename(p)).group(1)
        for p in glob.glob(os.path.join(OUT_DIR, "probs_daily", "gubapost_probs_*.parquet"))}
missing = expected - have
assert not missing, f"还有 {len(missing)} 个日文件未完成(缺 {min(missing)}~{max(missing)}), 先等推理跑完"

parts = []
t0 = time.time()
for p in sorted(glob.glob(os.path.join(OUT_DIR, "probs_daily", "gubapost_probs_*.parquet"))):
    df = pd.read_parquet(p, columns=["symbol", "available_date", "class_1_prob"])
    parts.append(df.groupby(["available_date", "symbol"], as_index=False)
                 .agg(sentiment_sum=("class_1_prob", "sum"),
                      n_posts=("class_1_prob", "count")))
acc = pd.concat(parts, ignore_index=True)
daily = (acc.groupby(["available_date", "symbol"], as_index=False)
         .agg(sentiment_sum=("sentiment_sum", "sum"), n_posts=("n_posts", "sum")))
daily["sentiment_mean"] = daily.sentiment_sum / daily.n_posts
daily = (daily[["available_date", "symbol", "sentiment_mean", "n_posts"]]
         .sort_values(["available_date", "symbol"]).reset_index(drop=True))

AGG_PATH = os.path.join(OUT_DIR, "gubapost_stockday_mean_" + "_".join(YEARS) + ".parquet")
daily.to_parquet(AGG_PATH, index=False)
print(f"聚合完成: {len(daily):,} 个 stock-day, 覆盖帖子 {daily.n_posts.sum():,} "
      f"({time.time()-t0:.0f}s) -> {AGG_PATH}")

In [ ]:
# 7. 聚合结果体检 ------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt

print("shape:", daily.shape)
print("日期范围:", daily.available_date.min(), "~", daily.available_date.max())
print("股票数:", daily.symbol.nunique(), "| 总帖数:", f"{daily.n_posts.sum():,}")
print("每日 stock-day 数: 中位", int(daily.groupby('available_date').size().median()),
      "| 最少", int(daily.groupby('available_date').size().min()))
print()
print("sentiment_mean 分布:")
print(daily.sentiment_mean.describe().to_string())
print()
print("n_posts 分布:")
print(daily.n_posts.describe(percentiles=[.5, .9, .99]).to_string())
assert daily.sentiment_mean.between(0, 1).all(), "sentiment_mean 越界!"

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ts = daily.groupby("available_date").sentiment_mean.mean()
axes[0].plot(range(len(ts)), ts.values, lw=0.8)
axes[0].set_title("全市场日度 sentiment_mean 时序")
axes[0].set_xlabel("交易日序号")
axes[1].hist(daily.sentiment_mean.sample(min(500_000, len(daily)), random_state=0), bins=60)
axes[1].set_title("stock-day sentiment_mean 直方图")
plt.tight_layout(); plt.show()

print("样例:")
print(daily.head(8).to_string(index=False))

## 产物清单

| 文件 | 内容 |
|---|---|
| `artifacts/gubapost_baseline/probs_daily/gubapost_probs_YYYYMMDD.parquet` | 逐帖推理结果(2020–2023) |
| `artifacts/gubapost_baseline/gubapost_stockday_mean_2020_2023.parquet` | stock-day 聚合(mean) |
| `artifacts/gubapost_baseline/inventory.parquet` | 输入文件清单与行数 |
| `artifacts/gubapost_baseline/manifest.json` | 推理配置与完成状态 |

后续回测可直接读 `gubapost_stockday_mean_*.parquet`, 以 `available_date` 为可知日,
`sentiment_mean` 为因子值。